# Oracles

Quantum oracles are a key part of many quantum algorithms that rely on quantum implementation of a classical function. The algorithms' discussions often assume that the quantum oracle that implements the function of interest is provided.  This kata dives deeper into the definition of different types of quantum oracles, their properties, and the basic ways to implement the oracles.

**This kata covers the following topics:**

- Quantum oracles and how they relate to classical oracles
- Two types of quantum oracles - phase oracles and marking oracles
- Phase kickback and its uses for implementing oracles
- Implementation and testing of quantum oracles in Workbench

**What you should know to start working on this kata:**

- Fundamental quantum concepts
- Basic multi-qubit gates, especially controlled gates

## Classical oracles

In classical computing, the terms "black box" versus "white box" are often used to discuss the testing of a function. In "white box" testing, the implementation of the function is visible to the tester, thus they can verify specific runtime or memory complexity expectations for the algorithm.

However, in "black box" testing, the tester doesn't have access to the details of the function implementation. They only have access to the "black box" that takes an input and produces the corresponding output. This means the tester can only test the functionality and expected behavior of the function, but not the implementation, which is hidden behind abstraction.

Formally, a **classical oracle** is a function that, provided some input, produces a *deterministic* output
(the same input *always* results in the same output).

Some classical problems (typically <a href="https://en.wikipedia.org/wiki/Decision_problem" target="_blank">decision problems</a>) are also expressed in terms of oracles; in this case the implementation of the function isn't important, but the functionality that it provides is.

> Suppose you've been given a function which checks whether a bit string matches the given pattern.
> This function is an example of a classical oracle.

In [ ]:
# Execute this cell to prepare the test infrastructure
from psiqdk.workbench import QPU, Qubits
from test_Oracles import problem

## Problem 1. Implement a classical oracle

**Input:** 
  A bit vector of length 3 represented as a `list[bool]` - a binary representation of a number in little-endian notation.

**Output:**
  Return `True` if the input array represents the number $7$, and `False` otherwise.

**Examples:**

* If the input array is `[True, True, True]`, return `True`.
* If the input array is `[True, True, False]`, return `False`.

In [ ]:
@problem
def is_seven(x: list[bool]) -> bool:
    # Write your code here
    ...

## Quantum oracles

An oracle in the quantum world is a "black box" operation that is used as input to an algorithm (such as Deutsch-Jozsa algorithm or Grover's search algorithm).
Many quantum algorithms take an oracle implementation of some classical function as input. They assume that such implementation is readily available for any function, but this is a very strong assumption - sometimes implementing the oracle for a function is a lot more complex than the algorithm that will use this oracle!  
In this kata, you'll learn the properties of quantum oracles and how to implement them.

A quantum oracle implements a function $f: \{0,1\}^n \rightarrow \{0,1\}^m$, where the input is $n$ bits of the form $x = (x_{0}, x_{1}, \dots, x_{n-1})$. In most commonly used cases $m=1$, that is, the function can return values $0$ or $1$. In this kata, you'll focus on this class of functions.

Quantum oracles operate on qubit registers (and can take additional classical parameters as well).  The classical input is encoded into the state of an $n$-qubit register:  
$$\ket{\vec{x}} = \ket{x_0} \otimes \ket{x_1} \otimes ... \otimes \ket{x_{n-1}},$$
where $\ket{x_i}$ represents the state of the $i$-th qubit.  

Oracles must be unitary transformations, and follow the same rules of linear algebra as other quantum operations.
This allows us to define quantum oracles based on their effect on the basis states - tensor products of single-qubit basis states $\ket{0}$ and $\ket{1}$.

> For example, an oracle that implements a function that takes two bits of input is defined using its effect on basis states $\ket{00}$, $\ket{01}$, $\ket{10}$, and $\ket{11}$.  

There are two types of quantum oracles: phase oracles and marking oracles.  Let's take a closer look at them.

## Phase oracles

For a function $f: \{0,1\}^n \rightarrow \{0,1\}$, the phase oracle $U_{\text{phase}}$ encodes the values of the function $f$ in the *relative phases* of basis states. When provided an input basis state $\ket{\vec{x}}$, it flips the sign of that state if $f(x)=1$:

$$U_{phase} \ket{\vec{x}} = (-1)^{f(x)}\ket{\vec{x}}$$

Thus, the phase oracle $U_{\text{phase}}$ doesn't change the phase of the basis states for which $f(x)=0$, but multiplies the phase of the basis states for which $f(x)=1$ by $-1$.

The effect of such an oracle on any single basis state isn't particularly interesting: it just adds a global phase, which is not something you can observe. However, if you apply this oracle to a *superposition* of basis states, its effect becomes noticeable.
Remember that quantum operations are linear: if you define the effect of an operation on the basis states, you'll be able to deduce its effect on superposition states (which are just linear combinations of the basis states) using its linearity.

A phase oracle doesn't have an "output", unlike the function it implements; the effect of the oracle application is the change in the state of the system.

## Demo: Phase oracle for alternating bit pattern function

Consider the function $f(x)$ that takes three bits of input and returns $1$ if $x=101$ or $x=010$, and $0$ otherwise.

The phase oracle that implements this function takes an array of three qubits as an input, flips the sign of basis states $\ket{101}$ and $\ket{010}$, and leaves the rest of the basis states unchanged. Let's see the effect of this oracle on a superposition state.

In [ ]:
# This operation implements the oracle; we will learn how to implement oracles later in the kata
def alternating_bit_pattern_phase_oracle(x : Qubits) -> None: 
    y = Qubits(1, "y", x.qpu)
    y.x()
    y.z(cond=x == 2)
    y.z(cond=x == 5)
    y.x()
    y.release()

qpu = QPU(num_qubits=4)
x = Qubits(3, "x", qpu=qpu)
x.had()

print("Starting state: an equal superposition of all basis states")
x.print_state_vector()

# Apply the oracle
alternating_bit_pattern_phase_oracle(x)

print("State after applying the phase oracle to mark |2⟩ and |5⟩")
_ = x.print_state_vector()

Notice that the input state in the demo above is an equal superposition of all basis states. After applying the oracle the absolute values of all amplitudes are the same, but the states $\ket{010}$ and $\ket{101}$ had their phase flipped to negative. Recall that these two states are exactly the inputs for which $f(x) = 1$, thus they're exactly the two states we expect to experience a phase flip!

## Problem 2. Implement a phase oracle for marking 7

**Input:**
  A Qubits register of length $3$ in an arbitrary superposition state $\ket{x}$ (the input register).

**Goal:**
Flip the sign of the input state $\ket{x}$ if the input register is in
the state $\ket{111}$ (encoding the integer $7$), and leave the input register unchanged otherwise.  
Don't allocate extra qubits to perform this operation.

**Examples:**

* If the input register is in the state $\ket{111}$, flip its sign.
* If the input register is in the state $\ket{010}$ or $\ket{101}$, do nothing.

<details>
  <summary><b>Need a hint?</b></summary>
  
  To solve this problem, you need to find a gate that will only flip the sign of the $\ket{111}$ basis state.  Which single-qubit gate flips the sign of the basis state $\ket{1}$ but not $\ket{0}$? How can you modify this gate to solve this problem?
</details>

In [ ]:
@problem
def is_seven_phase_oracle(x: Qubits) -> None:
    # Write your code here
    ...

## Marking oracles

A marking oracle $U_{mark}$ is an oracle that encodes the value of the classical function $f$ it implements in the *amplitude* of the qubit state. When provided an input array of qubits in the basis state $\ket{\vec{x}}$ and an output qubit in the basis state $\ket{y}$, it flips the state of the output qubit if $f(x)=1$. (You can also represent this as adding $f(x)$ to $y$ modulo $2$.) Hence, $U_{mark}$ is an operator that performs the following operation:

$$U_{mark}\ket{\vec{x}} \ket{y} = U_{mark}\big(\ket{\vec{x}} \otimes \ket{y}\big) = \ket{\vec{x}} \otimes \ket{y \oplus f(x)}$$

Again, since all quantum operations are linear, you can figure out the effect of this operation on superposition state knowing its effect on the basis states using its linearity.

A marking oracle has distinct "input" and "output" qubits, but in general the effect of the oracle application is the change in the state of the whole system rather than of the "output" qubits only. You'll look at this closer in a moment.

## Demo: Marking oracle for alternating bit pattern function

Consider the function $f(x)$ that takes three bits of input and returns $1$ if $x=101$ or $x=010$, and $0$ otherwise (it's the same function we used as an example of a phase oracle).

The marking oracle that implements this function takes a Qubits register of length $3$ as the "input" and a Qubits register of length $1$ as the "output", and flips the state of the output qubit if the input qubits are in basis state $\ket{101}$ or $\ket{010}$, doing nothing otherwise. Let's see the effect of this oracle on a superposition state.

In [ ]:
# This operation implements the oracle; we will learn how to implement oracles later in the kata
def alternating_bit_pattern_marking_oracle(x : Qubits, y : Qubits) -> None: 
    y.x(cond=x == 2)
    y.x(cond=x == 5)

qpu = QPU(num_qubits=4)
x = Qubits(3, "x", qpu=qpu)
y = Qubits(1, "y", qpu=qpu)

# Prepare an equal superposition
x.had()

print("Starting state: an equal superposition of all basis states on |x⟩")
qpu.print_state_vector()

# Apply the oracle
alternating_bit_pattern_marking_oracle(x,y)

print("State after applying the marking oracle to mark |2⟩ and |5⟩")
_ = qpu.print_state_vector()

Let's compare the initial state to the final state we saw in the demo above. The initial state is a tensor product of an equal superposition of all three-qubit basis states and the state $\ket{0}$.  For the final state, this is no longer the case. The basis states $\ket{010} \otimes \ket{0}$ and $\ket{101} \otimes \ket{0}$ have disappeared from the superposition, and instead the states $\ket{010} \otimes \ket{1}$ and $\ket{101} \otimes \ket{1}$ appeared.

This is exactly the expected result. Recall the function $f(x)$ that we implement here: $f(x)=1$ if and only if $x=010$ or $x=101$.  The first qubits register (variable `x`) represents the input state $\ket{x}$, and the last qubit (variable `y`) represents the output state $\ket{y}$.  Thus, for the two basis states $\ket{x}=\ket{010}$ and $\ket{x}=\ket{101}$, the function flips the state of the qubit $\ket{y}$, causing these two initial states to appear in the final state as a part of a tensor product with $\ket{1}$, where originally they were a part of a tensor product with $\ket{0}$.

Since the rest of the basis states correspond to $f(x) = 0$, all other basis states in the initial superposition remain unchanged.

## Problem 3. Implement a marking oracle for marking 7

**Inputs:**
  1. A Qubits register of length $3$ in an arbitrary superposition state $\ket{x}$ (the input register).
  2. A Qubits register of length $1$ in an arbitrary superposition state $\ket{y}$ (the output register).

**Goal:**
Flip the state of $\ket{y}$ if the input register is in the state $\ket{111}$, and leave the state $\ket{y}$ unchanged otherwise.

**Examples:**

* If the query register is in the state $\ket{111}$, flip the state of the output qubit $\ket{y}$.
* If the query register is in the state $\ket{010}$ or $\ket{101}$, do nothing.

In [ ]:
@problem
def is_seven_marking_oracle(x: Qubits, y: Qubits) -> None:
    # Write your code here
    ...

## Phase kickback

Previously we considered applying marking oracles when the register $\ket{x}$ was in a basis state or a superposition state, and the "output" qubit $\ket{y}$ was in a basis state.  How might the effect of applying marking oracles change if the "output" is also in a superposition state?  In this case we might observe **phase kickback** - the relative phase from the "output" qubit affecting ("kicked back" into) the state of the input qubits.

In order to observe phase kickback when applying a marking oracle, let's use the "output" qubit in the state $\ket{y} = \ket{-}$.

> This is the standard choice for two reasons.
> First, for phase kickback to occur, the "output" qubit must have a difference in relative phase between the basis states $\ket{0}$ and $\ket{1}$.
> Second, the absolute values of the amplitudes of the two basis states of the "output" qubit must be equal, otherwise that qubit will become entangled with the input register, and we want to avoid this.

Let's see the results of applying a marking oracle $U_{mark}$ which implements the function $f(x)$ to the input register $\ket{x}$ and the "output" qubit in state $\ket{-}$:

What happens if the input register $\ket{x}$ is in a basis state?

$$U_{mark} \ket{x} \ket{-} = \tfrac1{\sqrt2} \big(U_{mark}\ket{x}\ket{0} - U_{mark}\ket{x} \ket{1}\big) =$$

$$= \tfrac1{\sqrt2} \big(\ket{x}\ket{0\oplus f(x)} - \ket{x} \ket{1\oplus f(x)}\big) =$$

$$=\begin{cases}
\tfrac1{\sqrt2} \big(\ket{x}\ket{0} - \ket{x} \ket{1}\big) = \ket{x}\ket{-} \text{ if } f(x) = 0 \\
\tfrac1{\sqrt2} \big(\ket{x}\ket{1} - \ket{x} \ket{0}\big) = -\ket{x}\ket{-} \text{ if } f(x) = 1
\end{cases}=$$

$$= (-1)^{f(x)}\ket{x} \ket{-}$$

Now, let's say that the input register is in a superposition state, for example, $\ket{x} = \frac1{\sqrt2} \big(\ket{b_1} + \ket{b_2}\big)$, where $\ket{b_1}$ and $\ket{b_2}$ are basis states. Then

$$U_{mark} \ket{x} \ket{-} = U_{mark} \tfrac{1}{\sqrt{2}} \big(\ket{b_1} + \ket{b_2}\big) \ket{-} =$$

$$= \tfrac{1}{\sqrt{2}} \big( U_{mark}\ket{b_1}\ket{-} + U_{mark}\ket{b_2}\ket{-}\big) =$$

$$= \tfrac{1}{\sqrt{2}} \big( (-1)^{f(b_1)}\ket{b_1} + (-1)^{f(b_2)}\ket{b_2}\big) \ket{-}$$

In both cases applying $U_{mark}$ doesn't change the state of the "output" qubit, but it does change the state of the input register.
Thus, you can drop the "output" qubit after you applied the oracle without any repercussions.

### Using phase kickback to convert marking oracle into a phase oracle

Notice that after applying the marking oracle to the example superposition state the input register ended up in the following state:

$$\ket{\psi} = \tfrac{1}{\sqrt{2}} \big( (-1)^{f(b_1)}\ket{b_1} + (-1)^{f(b_2)}\ket{b_2}\big)$$

This looks exactly as if you apply a *phase* oracle to $\ket{x}$ instead of applying a *marking* oracle to $\ket{x}\ket{-}$!  This is a very important application of phase kickback: it allows to convert a marking oracle into a phase oracle - which you'll implement in the next task.

> Another important application of the phase kickback effect is **phase estimation** algorithm, which allows to estimate an eigenvalue of an eigenvector. We will discuss it in a later kata.

### Phase kickback: specific oracle example

Consider the following example using the $U_{7,mark}$ oracle. Let's begin with $\ket{x}$ as an equal superposition of the $\ket{110}$ and $\ket{111}$ basis states and $\ket{y}=\ket{-}$. The overall initial state is:

$$\ket{\eta} = \Big[\tfrac{1}{\sqrt{2}}\big(\ket{110} + \ket{111}\big)\Big] \otimes \tfrac{1}{\sqrt{2}}\big(\ket{0} - \ket{1}\big) =$$

$$= \tfrac{1}{2} \big(\ket{110}\ket{0} + \ket{111}\ket{0} - \ket{110}\ket{1} - \ket{111}\ket{1}\big)$$

How does $U_{7,mark}$ act on this state?

$$U_{7,mark}\ket{\eta} = U_{7,mark} \tfrac{1}{2} \big(\ket{110}\ket{0} + \ket{111}\ket{0} - \ket{110}\ket{1} - \ket{111}\ket{1} \big) =$$

$$= \tfrac{1}{2} \big( U_{7,mark}\ket{110}\ket{0} + U_{7,mark}\ket{111}\ket{0} - U_{7,mark}\ket{110}\ket{1} - U_{7,mark}\ket{111}\ket{1} \big) =$$

$$= \tfrac{1}{2} \big(\ket{110}\ket{0} + \ket{111}\ket{1} - \ket{110}\ket{1} - \ket{111}\ket{0} \big) := \ket{\xi}$$

Now let's see how the input state $\ket{\eta}$ is modified by the oracle.  Let's simplify the resulting state $\ket{\xi}$:

$$\ket{\xi} = \tfrac{1}{2} \big(\ket{110}\ket{0} + \ket{111}\ket{1} - \ket{110}\ket{1} - \ket{111}\ket{0}\big) =$$

$$= \tfrac{1}{2} \big(\ket{110}\ket{0} - \ket{110}\ket{1} - \ket{111}\ket{0} + \ket{111}\ket{1} \big) =$$

$$= \tfrac{1}{2} \Big[\ket{110} \otimes \big(\ket{0} - \ket{1} \big) + \ket{111} \otimes \big(\ket{1} - \ket{0}\big)\Big] =$$

$$= \Big[\tfrac{1}{\sqrt{2}} \big( \ket{110} - \ket{111} \big) \Big] \otimes \Big[ \tfrac{1}{\sqrt{2}} \big( \ket{0} - \ket{1} \big) \Big] =$$

$$= \Big[\tfrac{1}{\sqrt{2}} \big( \ket{110} - \ket{111} \big) \Big] \otimes \ket{-}$$

Finally, let's compare $\ket{\eta}$ and $\ket{\xi}$; below are the final equations repeated for your convenience:

$$\ket{\eta} = \Big[\tfrac{1}{\sqrt{2}}\big(\ket{110} + \ket{111}\big)\Big] \otimes \ket{-}$$
$$\ket{\xi} = \Big[\tfrac{1}{\sqrt{2}}\big(\ket{110} - \ket{111}\big)\Big] \otimes \ket{-}$$

You can see that these two equations are identical, except for the $-1$ phase that appeared on the $\ket{111}$ basis state - our marked state.  This is a specific example of the phase kickback effect, as the phase from $\ket{-}$ has been *kicked back* into $\ket{x}$.


## Problem 4. Implement a marking oracle as a phase oracle

**Inputs:**
  1. A marking oracle implementing an unknown $N$-bit function $f(x)$ as a Workbench function.
  2. A Qubits register of length $3$ in an arbitrary superposition state $\ket{x}$ (the input register).
  
**Goal:**
Flip the phase of each basis state $\ket{x}$ for which $f(x) = 1$. You can only access $f(x)$ via the marking oracle you are given.

<details>
  <summary><b>Need a hint?</b></summary>
    Recall that you can allocate extra qubits to assist in this operation.  Is there a state that you could prepare on an auxiliary qubit which would help you to convert the marking oracle to a phase oracle?
</details>

In [ ]:
@problem
def apply_marking_oracle_as_phase_oracle(marking_oracle: callable, x: Qubits) -> None:
    # You can allocate a qubit in a Workbench function using the following syntax:
    # q = Qubits(1, "q", x.qpu)

    # Write your code here
    ...

    # Remember to release any qubits you allocate at the end of the function!
    # q.release()

## Demo: Converting marking oracles to phase oracles

This demo uses a reference implementation of `apply_marking_oracle_as_phase_oracle` operation to convert marking oracle `is_seven_marking_oracle` to a phase oracle. You can compare this converted oracle to the reference implementation of the phase oracle `is_seven_phase_oracle`. You've already implemented both these oracles in the previous tasks.

In [ ]:
def is_seven_phase_oracle(x: Qubits) -> None:
    x[0].z(cond=x[1:])

def is_seven_marking_oracle(x: Qubits, y: Qubits) -> None:
    y.x(cond = x == 7)

def apply_marking_oracle_as_phase_oracle(marking_oracle, x: Qubits) -> None:
    y = Qubits(1, "y", x.qpu)
    y.x()
    y.had()
    marking_oracle(x, y)
    y.had()
    y.x()
    y.release()

qpu = QPU(num_qubits=4)
x = Qubits(3, "x", qpu=qpu)
x.had()
print("Starting state: an equal superposition of all basis states on |x⟩")
x.print_state_vector()

# Apply the phase oracle
is_seven_phase_oracle(x)

print("The state after applying the phase oracle is_seven_phase_oracle")
x.print_state_vector()

# Reset the register and recreate an equal superposition of all basis states
x.write(0)
x.had()

# Apply the marking oracle as a phase oracle
apply_marking_oracle_as_phase_oracle(is_seven_marking_oracle, x)

print("The state after applying the converted marking oracle is_seven_marking_oracle")
_ = x.print_state_vector()

This demo shows that the phase oracle $U_{7,phase}$ behaves the same as the converted version of the marking oracle $U_{7,mark}$: both induce a phase flip on the basis state $\ket{111}$!

This way to convert a marking oracle to a phase oracle is useful because many quantum algorithms, such as Grover's search algorithm, rely on a phase oracle, but it's often easier to implement the function as a marking oracle. Phase kickback converter provides a way to implement the function of interest as a marking oracle and then convert it into a phase oracle, which can then be leveraged in a quantum algorithm.

## Practice implementing quantum oracles

In the next section of this kata you'll implement a few more complicated quantum oracles, both phase and marking. Some of them can take extra "classical" parameters. A useful tool for implementing quantum oracles is allocating auxiliary qubits to assist in a computation. You'll practice that in some of the exercises below.

## Problem 5. Implement the OR oracle

**Inputs:**
  1. A Qubits register of length $N$ in an arbitrary superposition state $\ket{x}$ (the input register).
  2. A Qubits register of length $1$ in an arbitrary superposition state $\ket{y}$ (the output register).

**Goal:**
Flip the state of $\ket{y}$ if the input register is in any basis state
except for $\ket{00...0}$ (the all-zero state).

**Examples:**

* If the input register is in the state $\ket{10000001}$, $\ket{11101101}$ or $\ket{0010101}$, flip the state $\ket{y}$.
* If the input register is in the state $\ket{000}$, do nothing.

<details>
  <summary><b>Before implementing this oracle, answer the question: are you implementing a marking or a phase oracle?</b></summary>
  
  This is a marking oracle, because you're flipping the state of the target qubit $\ket{y}$ based on the state of the input $\ket{x}$.
</details>

<br/>
<details>
  <summary><b>Need a hint?</b></summary>

  You need to flip the state of $\ket{y}$ for every input except $\ket{00...0}$, or, alternatively, flip it unconditionally and then flip it for the $\ket{00...0}$ state.
</details>

In [ ]:
@problem
def or_oracle(x: Qubits, y: Qubits) -> None:
    # Write your code here
    ...

## Problem 6. Implement the K-th bit oracle

**Inputs:**
  1. A Qubits register of length $N$ in an arbitrary superposition state $\ket{x}$ (the input register).
  2. An integer $k$ such that $0 \leq k < N$.

**Goal:**
Flip the sign of the input state $\ket{x}$ if the $k$-th bit of $x$ is $1$.  
*Implement this oracle without using auxiliary qubits.*

**Examples:**

* If the input register is in the state $\ket{010}$ and $k=0$, do nothing.
* If the input register is in the state $\ket{010}$ and $k=1$, flip the sign of the basis state.

<details>
  <summary><b>Before implementing this oracle, answer the question: are you implementing a marking or a phase oracle?</b></summary>

  This is a phase oracle, because you're changing the phase of the input state $\ket{x}$ based on the value of the function $f(x)$.
</details>

In [ ]:
@problem
def kth_bit_oracle(x: Qubits, k: int) -> None:
    # Write your code here
    ...

## Problem 7. Implement the OR oracle of all bits except the K-th

**Inputs:**
  1. A Qubits register of length $N \geq 2$ in an arbitrary superposition state $\ket{x}$ (the input register).
  2. An integer $k$ such that $0 \leq k < N$.

**Goal:**
Flip the sign of the basis state $\ket{x}$ if any of the bits of $x$ (not considering the $k$-th bit) are $1$ in the input register. In other words, the input register with the $k$-th qubit excluded should be in any state except the all-zero state to flip the sign of the input register. The state of the $k$-th qubit does not affect the result.

*Feel free to explore implementing this operation with or without auxiliary qubits.*

**Examples:**

* If the input register is in the state $\ket{010}$ and $k=0$, flip the sign of the register.
* If the input register is in the state $\ket{010}$ and $k=1$, do nothing.

<details>
  <summary><b>Before implementing this oracle, answer the question: are you implementing a marking or a phase oracle?</b></summary>
  
  This is a phase oracle, because you're changing the phase of the input state $\ket{x}$ based on the value of the function $f(x)$.
</details>

<br/>
<details>
  <summary><b>Need a hint?</b></summary>
  
  You can use the previously implemented oracles if needed.
  <br/>You can use Qubits register slicing (similar to Python slicing) to get parts of the array before and after the $k$-th element.
</details>

In [ ]:
@problem
def or_of_bits_except_kth_oracle(x: Qubits, k: int) -> None:
    # Write your code here
    ...

## Problem 8. Implement the arbitrary bit pattern oracle

**Inputs:**
  1. A Qubits register of length $N$ in an arbitrary superposition state $\ket{x}$ (the input register).
  2. A Qubits register of length $1$ in an arbitrary superposition state $\ket{y}$ (the output register).
  3. A bit string `pattern` of length $N$ represented as a `list[bool]` encoding a basis state. Values `True` and `False` correspond to $\ket{1}$ and $\ket{0}$, respectively.

**Goal:**
Flip the state of $\ket{y}$ if the input register matches the basis state represented by `pattern`.  

**Examples:**

* If the input register is in the state $\ket{010}$ and `pattern = [False, True, False]`, flip the state $\ket{y}$.
* If the input register is in the state $\ket{1001}$ and `pattern = [False, True, True, False]`, do nothing.
    
<br/>
<details>
  <summary><b>Before implementing this oracle, answer the question: are you implementing a marking or a phase oracle?</b></summary>
  
  This is a marking oracle, because you're flipping the state of the target qubit $\ket{y}$ based on the state of the input $\ket{x}$.
</details>

<br/>
<details>
  <summary><b>Need a hint?</b></summary>
  
  You need to flip the state of $\ket{y}$ if $\ket{x}$ matches the given pattern. Remember that Workbench allows you to apply controlled gates with arbitrary control patterns in the `cond` argument.
</details>

In [ ]:
@problem
def arbitrary_bit_pattern_oracle(x: Qubits, y: Qubits, pattern: list[bool]) -> None:
    # Write your code here
    ...

## Problem 9. Implement the arbitrary bit pattern oracle (challenge version)

**Inputs:**
  1. A Qubits register of length $N$ in an arbitrary superposition state $\ket{x}$ (the input register).
  2. A bit string `pattern` of length $N$ represented as a `list[bool]` encoding a basis state. Values `True` and `False` correspond to $\ket{1}$ and $\ket{0}$, respectively.

**Goal:**
Flip the sign of the input state $\ket{x}$ if the input register matches the basis state represented by `pattern`.  
*Implement this oracle without using auxiliary qubits*

**Examples:**

 * If the input register is in the state $\ket{010}$ and `pattern = [False, True, False]`, flip the sign of the input register.
 * If the input register is in the state $\ket{1001}$ and `pattern = [False, True, True, False]`, do nothing.
  
<br/>
<details>
  <summary><b>Before implementing this oracle, answer the question: are you implementing a marking or a phase oracle?</b></summary>
  
  This is a phase oracle, because you're changing the phase of the input state $\ket{x}$ based on the value of the function $f(x)$.
</details>

<br/>
<details>
  <summary><b>Need a hint?</b></summary>
  
  Can you transform the state of the input register based on the <code>pattern</code> value so that you can accomplish the task by flipping the phase only for the $\ket{1...1}$ state?

  Alternatively, consider using the <code>reflect</code> method of Qubits class which multiples one basis state by a given relative phase.
</details>

In [ ]:
@problem
def arbitrary_bit_pattern_oracle_challenge(x: Qubits, pattern: list[bool]) -> None:
    # Write your code here
    ...

## Demo: Testing an oracle implementation

This demo shows how to test an oracle that you've implemented for your own problem.
For all oracles you've implemented in this kata, you've been testing your oracle using the tests already implemented for you.
However, if you're designing an oracle for a new problem, you don't have a ready-made test for it; you have to design and implement one yourself.

A good way to test a marking oracle is to write a classical oracle (a Python function) that performs the same computation classically, and then compare the effect of your marking oracle on the basis states with the output of the classical oracle for every input (or a certain percentage of the inputs if you are constrained by runtime) to ensure that they match.

The following demo shows you how to compare the implementation of the OR oracle to the classical code implementing the same function. To make this comparison efficient, you'll want to use Workbench [bit vector simulator](https://docs.construct.psiquantum.com/workbench/new-tutorials/Simulating-WB-Programs.html#bit-vector-simulator) (filter `>>bit-sim>>`, configured in filter preset `BIT_DEFAULT`), designed specifically for simulating reversible computations.

In [ ]:
from psiqdk.workbench import QPU, Qubits
from psiqdk.workbench.filter_presets import BIT_DEFAULT

def or_oracle(x: Qubits, y: Qubits) -> None:
    '''The marking oracle you want to test.'''
    y.x(cond=x == 0)
    y.x()

def f_or(args: list[bool]) -> bool:
    '''The classical function that performs the same computation as the marking oracle should.'''
    return any(args)


def test_or_oracle() -> None:
    '''The test that compares the marking oracle with the classical function on a variety of test cases.'''
    # Define a QPU instance to run the test on
    qpu = QPU(filters=BIT_DEFAULT)

    # Let's test functions acting on inputs of different sizes.
    for n in range(2, 5):
        # There are 2^n possible basis states for each n. 
        # Our n is very small, so let's test all possible basis states.
        for input_int in range(2 ** n):
            # Reset the QPU for each individual test.
            qpu.reset(n + 1)
            # Allocate quantum registers we'll use.
            x = Qubits(n, "x", qpu)
            y = Qubits(1, "y", qpu)

            # Initialize input qubits to the basis state we're testing.
            x.write(input_int)

            # Apply the marking oracle.
            or_oracle(x, y)

            # Read out the state of the qubits after the quantum computation.
            x_actual = x.read()
            y_actual = y.read()

            # Convert input_mask to an array of Boolean values.
            input_bits = [bool(input_int & (1 << i)) for i in range(n)]

            # Calculate the result of the classical function.
            y_expected = f_or(input_bits)

            # Check that the actual values match the expected ones.
            if y_actual != y_expected:
                raise Exception(f"Error for input={input_bits}: expected result {y_expected}, got {y_actual}")
            if x_actual != input_int:
                raise Exception(f"Error for input={input_bits}: the state of the input qubits changed")

    print("Test passed!")

test_or_oracle()

## Conclusion

Congratulations! In this kata you've learned to build quantum oracles. Here are a few key concepts to keep in mind:

- A quantum oracle is an "opaque box" operation that implements a classical computation.
- Quantum oracles are used to convert classical problems into inputs to quantum algorithms, such as Grover's search algorithm.
- Phase oracles encode the information in the relative phase of basis states. If $f(x)=0$, the oracle doesn't change the basis state $\ket{x}$, and if $f(x)=1$, it multiplies the phase of the basis state $\ket{x}$ by $-1$.
- Marking oracles use an extra qubit $\ket{y}$ and encode the information in the state of that qubit. If $f(x)=0$, the oracle doesn't change the state of the qubit $\ket{y}$ for the basis state $\ket{x}$, and if $f(x)=1$, it flips the state of the qubit $\ket{y}$ for the basis state $\ket{x}$.

> Copyright (c) 2026 PsiQuantum